# 33. A classifier per band of widths, on A and K

Two branches, one per card, over more than one session - see below for
how to carry it forward. Nothing in this notebook needs editing: run it as it is.

A and K on ResNet-50 again, each with **one classifier per band of
widths** instead of one classifier for every width. Everything else is BX
and BY from notebook 30, so **notebook 30 has to come back first**: CH is
read against BX and CI against BY.

**Where it comes from.** SOLAR (WACV 2026) gives every subnet of a
once-for-all network its own classifier over the shared backbone and
reports better accuracy and better calibration: the subnets stop fighting
over one set of logit weights. Its subnets are a fixed eight. Here the
width is continuous, so one head per width is impossible. `head_groups: 4`
cuts 0.25 to 1.00 into four equal bands - at the test widths 0.25-0.40,
0.45-0.60, 0.65-0.80, 0.85-1.00, four to a band - and each band has its
own head. The network still runs at any width. This is an adaptation of
SOLAR, not SOLAR, and should be written up that way.

**What else it touches.** Nothing. The widest band's head keeps the name
`classifier`, so the teacher, the class cost matrix and the state dict of
every earlier branch are unchanged; `head_groups` defaults to 1, which
builds the model it always did. The three extra heads add 0.6M parameters
to 23.7M. The two middle heads train only on steps where a free width
lands in their band, about 44 per cent of steps each.

**What was checked.** tests/test_head_groups.py: the sixteen test widths
fall four to a band, a backward at 0.25, 0.50, 0.70 and 1.00 reaches its
own head and no other, width 0.25 reads a different head from 1.00, and
the class cost matrix still reads the widest head. With the head selection
broken on purpose it fails. Peak memory at batch 256: CH 9.47 GB, CI
9.38 GB, both under the 15 GB of a T4.

**How to read it.** Two questions. Does a head per band help A - if so the
shared classifier was a bottleneck. And does it add to K, or does K's
transport already relieve the same thing. Look at NLL as well as top-1:
SOLAR's claim is calibration, and so is K's clearest one.

## K will need a second session

Notebook 30 measured A at 10.7 hours and K at about 8 minutes an epoch,
about 14 hours, so CH finishes in one session and CI will not. When the session ends, attach
its output, set `RESUME_FROM` and run again; CH will calibrate, print its
table and exit in a minute.

## `ch_a_heads4_r50`

A on ResNet-50, with head_groups: 4. Identical to BX otherwise. From SOLAR (WACV 2026), which gives every subnet of a once-for-all network its own classifier over the shared backbone and reports better accuracy and calibration: the subnets stop fighting over one set of logit weights. SOLAR's subnets are a fixed eight. Here the width is continuous, so there cannot be one head per width. The range 0.25 to 1.00 is cut into four equal bands, 0.25-0.40, 0.45-0.60, 0.65-0.80 and 0.85-1.00 at the test widths, four test widths to a band, and every width in a band reads that band's head. This is an adaptation, not SOLAR: it keeps the network runnable at any width, which one head per test width would not. The widest band's head keeps the name classifier, so the teacher, the class cost matrix and every other reader of the classifier see the widest width's head. The backbone, the sampler and the loss are BX's. The three extra heads, 2048 by 100 each, add 0.6M parameters to 23.7M. The sandwich rule always runs 0.25 and 1.00, so the first and last heads train every step; the two middle heads train only when a free width lands in their band, about 44 per cent of steps each. Their rows see fewer updates, which is part of what is being tested. Read CH against BX. If the heads help A, the shared classifier was a bottleneck; if they help A and not K, the transport term was already relieving it.

## `ci_k_heads4_r50`

K on ResNet-50, with head_groups: 4. Identical to BY otherwise. From SOLAR (WACV 2026), which gives every subnet of a once-for-all network its own classifier over the shared backbone and reports better accuracy and calibration: the subnets stop fighting over one set of logit weights. SOLAR's subnets are a fixed eight. Here the width is continuous, so there cannot be one head per width. The range 0.25 to 1.00 is cut into four equal bands, 0.25-0.40, 0.45-0.60, 0.65-0.80 and 0.85-1.00 at the test widths, four test widths to a band, and every width in a band reads that band's head. This is an adaptation, not SOLAR: it keeps the network runnable at any width, which one head per test width would not. The widest band's head keeps the name classifier, so the teacher, the class cost matrix and every other reader of the classifier see the widest width's head. The backbone, the sampler and the loss are BY's. The three extra heads, 2048 by 100 each, add 0.6M parameters to 23.7M. The sandwich rule always runs 0.25 and 1.00, so the first and last heads train every step; the two middle heads train only when a free width lands in their band, about 44 per cent of steps each. Their rows see fewer updates, which is part of what is being tested. Read CI against BY. If CI beats BY, the heads and the transport add up. The NLL column matters as much as top-1: SOLAR claims calibration, and so does K.


## Before you start

* Accelerator **GPU T4 x2**, Internet **On**
* Add the CIFAR-100 dataset as an input
* **Save Version -> Save & Run All (Commit)**, not the interactive run.
  An interactive session ends when the browser closes.

The checks at the top run on the CPU and stop the session in about two
minutes if anything is wrong, before a card is touched. Note that the
branch suite mimics the training loop rather than running it, so for
these branches the smoke step below is what actually exercises the new
code. Do not skip it.

If the session times out partway, carry it forward rather than starting
over. Every epoch writes a checkpoint, so at most one is lost, and the
learning-rate schedule is restored with it - resuming does not restart
the cosine.

1. Save Version on the timed-out session, so its output persists.
2. Make a fresh copy of this notebook.
3. Add data -> Notebook Output -> pick that session.
4. Set `RESUME_FROM` to the directory that *contains* `cifar100_<branch>`,
   then Save & Run All.

That directory is in one of two places depending on how the session
ended, because the copy at the bottom of this notebook only runs if the
session got that far:

    finished the last cell   /kaggle/input/<slug>/logs
    killed mid-training      /kaggle/input/<slug>/new-pruning/logs

If in doubt, run `!find /kaggle/input -name latest_checkpoint.pt` in a
scratch cell and take the parent of the parent. A correct path prints
`restored cifar100_<branch>` and then `Loaded checkpoint ... at epoch N`;
if you see `starting from scratch` instead, the path is wrong and the run
will silently begin again at epoch 1.

## When it finishes

Send back the final table. Pasting the output of the last cell is enough.

In [ ]:
# Fixed for this notebook. Notebook 33 of 38.
BRANCHES = ['ch_a_heads4_r50', 'ci_k_heads4_r50']

SMOKE_FIRST = True

REPO_URL = 'https://github.com/duyh80456-code/new-pruning.git'
REPO_BRANCH = 'nhan'

CIFAR_DIR = ('/kaggle/input/datasets/nlnk1607/cifar100/cifar-100-python')

# To carry a timed-out session forward, attach its output and name the
# logs directory. Leave empty to start fresh.
RESUME_FROM = ''

In [ ]:
import os
import queue
import re
import shutil
import subprocess
import sys
import threading
import time

import torch

n_gpu = torch.cuda.device_count()
print('torch', torch.__version__, '| gpus', n_gpu)
for i in range(n_gpu):
    print('  {}: {}'.format(i, torch.cuda.get_device_properties(i).name))

WORK = '/kaggle/working'
CODE = os.path.join(WORK, 'new-pruning')
if not os.path.isdir(CODE):
    subprocess.run(
        ['git', 'clone', '-b', REPO_BRANCH, REPO_URL, CODE], check=True)
os.chdir(CODE)

available = sorted(
    name[len('cifar100_'):-len('.yml')]
    for name in os.listdir('apps')
    if name.startswith('cifar100_') and name.endswith('.yml'))
print('\nbranches in apps/:')
for name in available:
    print('   ', name)

missing = [b for b in BRANCHES if b not in available]
if missing:
    raise SystemExit('no config for {}'.format(missing))
if n_gpu == 0:
    raise SystemExit('No GPU in this session. Set Accelerator to '
                     'GPU T4 x2 in the notebook settings and run again.')
if n_gpu < len(BRANCHES):
    print('\n{} branches, {} gpu(s): they will run in sequence.'.format(
        len(BRANCHES), n_gpu))

In [ ]:
# What the chosen branches actually differ in, read off the configs
# rather than from the table above, which can drift.
AXES = ('kd_loss', 'cost_source', 'feature_kd', 'feature_align',
        'feature_layers', 'feature_weight', 'tier_weights',
        'horizontal_kd', 'horizontal_where', 'horizontal_loss',
        'horizontal_weight', 'weight_schedule')

settings = {}
for branch in BRANCHES:
    found = {}
    with open('apps/cifar100_{}.yml'.format(branch)) as handle:
        for line in handle:
            key = line.split(':')[0].strip()
            if key in AXES:
                found[key] = line.split(':', 1)[1].strip()
    settings[branch] = found

print('{:20}'.format('') + ''.join(
    '{:>26}'.format(b) for b in BRANCHES))
for axis in AXES:
    values = [settings[b].get(axis, '-') for b in BRANCHES]
    if all(v == '-' for v in values):
        continue
    print('{:20}'.format(axis) + ''.join(
        '{:>26}'.format(v) for v in values))

In [ ]:
TARGET = 'data/cifar-100-python'
if not os.path.isdir(TARGET):
    source = CIFAR_DIR if os.path.isdir(CIFAR_DIR) else None
    if source is None:
        for root, dirs, _ in os.walk('/kaggle/input'):
            if 'cifar-100-python' in dirs:
                source = os.path.join(root, 'cifar-100-python')
                break
    os.makedirs('data', exist_ok=True)
    if source:
        os.symlink(source, TARGET)
        print('linked', source)
    else:
        from torchvision import datasets
        datasets.CIFAR100(root='data', train=True, download=True)
        datasets.CIFAR100(root='data', train=False, download=True)
print(sorted(os.listdir(TARGET)))

## Checks, before a card is touched

Seconds on the CPU. Between them these suites have caught a Sinkhorn solved too loosely to have a correct gradient, an alpha-divergence that destroyed the weights in three steps, a calibration that reset the batch norm statistics and never refilled them, and a feature cost normalized so that its own gradient vanished. Every one of those was silent.

The last suite builds each branch in this notebook and runs two training steps of the real loop, profiling included. Four feature branches once reached Kaggle, passed every loss check, and died in the profiler on the first forward, because nothing local had ever called it.

In [ ]:
# Driven by BRANCHES, not by a list written beside it. The branch suite
# used to name its two branches literally, which quietly checked the wrong
# pair for anyone who changed BRANCHES and nothing else.
suites = ['tests/test_loss_ops.py', 'tests/test_bn_calibration.py',
          'tests/test_kd_variants.py',
          'tests/test_channel_reorder.py',
          'tests/test_published_methods.py',
          'tests/test_freeze.py',
          'tests/test_head_groups.py',
          'tests/test_affine_knots.py',
          'tests/test_no_kd.py']
for suite in suites:
    done = subprocess.run([sys.executable, suite], capture_output=True,
                          text=True)
    print('{:34} {}'.format(
        suite, (done.stdout.strip().splitlines() or ['no output'])[-1]))
    if done.returncode != 0:
        print(done.stdout[-3000:], done.stderr[-2000:])
        raise SystemExit('{} failed'.format(suite))

done = subprocess.run(
    [sys.executable, 'tests/test_all_branches.py'] + BRANCHES,
    capture_output=True, text=True)
print()
print(done.stdout.strip()[-2000:])
if done.returncode != 0:
    print(done.stderr[-2000:])
    raise SystemExit('a branch in {} does not build'.format(BRANCHES))

In [ ]:
if RESUME_FROM:
    os.makedirs('logs', exist_ok=True)
    for name in os.listdir(RESUME_FROM):
        src = os.path.join(RESUME_FROM, name)
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join('logs', name),
                            dirs_exist_ok=True)
            print('restored', name)
else:
    print('starting from scratch')

In [ ]:
VAL_LINE = re.compile(
    r'val\s+([0-9.]+)\s+-1/\d+:\s+loss:\s+([0-9.eE+-]+),\s+'
    r'top1_error:\s+([0-9.]+)')
results = {}


def run_pinned(jobs, quiet=True):
    """one job per card, output interleaved and tagged"""
    lines = queue.Queue()
    procs = {}
    failing = set()

    def pump(label, proc):
        for line in proc.stdout:
            lines.put((label, line.rstrip('\n')))
        proc.wait()
        lines.put((label, None))

    for index, (label, config) in enumerate(jobs):
        env = dict(os.environ)
        env['CUDA_VISIBLE_DEVICES'] = str(index % max(n_gpu, 1))
        proc = subprocess.Popen(
            [sys.executable, '-u', 'train.py', 'app:' + config],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, env=env)
        procs[label] = proc
        threading.Thread(target=pump, args=(label, proc),
                         daemon=True).start()
        print('[{}] started on gpu {} with {}'.format(
            label, env['CUDA_VISIBLE_DEVICES'], config), flush=True)

    started = time.time()
    remaining = len(jobs)
    while remaining:
        label, line = lines.get()
        if line is None:
            remaining -= 1
            print('[{}] exit code {} after {:.0f} min'.format(
                label, procs[label].returncode,
                (time.time() - started) / 60), flush=True)
            continue
        found = VAL_LINE.search(line)
        if found:
            width, loss, top1 = found.groups()
            results.setdefault(label, {})[float(width)] = (
                float(loss), float(top1))
        # Once a branch starts printing a traceback, stop filtering
        # it. The body is indented, so the rule below would keep the
        # exception and throw away where it came from.
        if 'Traceback (most recent call last)' in line:
            failing.add(label)
        if quiet and label not in failing and (
                line.startswith(('  ', ')', 'Model(', 'Total', 'Item'))
                or not line.strip()):
            continue
        print('[{}] {}'.format(label, line), flush=True)

    return {label: proc.returncode for label, proc in procs.items()}

In [ ]:
if SMOKE_FIRST:
    codes = run_pinned(
        [(b, 'apps/smoke_{}.yml'.format(b)) for b in BRANCHES])
    failed = [b for b, code in codes.items() if code != 0]
    if failed:
        raise SystemExit('smoke failed for {}'.format(failed))
    for b in BRANCHES:
        shutil.rmtree('logs/smoke_{}'.format(b), ignore_errors=True)
    results.clear()
    print('\nsmoke ok')

In [ ]:
codes = run_pinned(
    [(b, 'apps/cifar100_{}.yml'.format(b)) for b in BRANCHES])
print(codes)

## Results, against A

A is the number to beat, not C or D. Improving on an ablation of your own
method is not improving on the paper.

One seed, and sigma has not been measured. Three of the four gaps in the
finished table sit between 0.25 and 0.44 points, which is the range where
a single run cannot tell a result from noise.

In [ ]:
# the published run, for reference
A_KL = {0.25: 70.10, 0.30: 70.80, 0.35: 71.50, 0.40: 72.20, 0.45: 72.80,
        0.50: 73.20, 0.55: 73.40, 0.60: 73.80, 0.65: 73.90, 0.70: 74.30,
        0.75: 74.60, 0.80: 74.80, 0.85: 75.10, 0.90: 75.10, 0.95: 75.40,
        1.00: 75.30}

widths = sorted({w for table in results.values() for w in table})
header = '{:>7}{:>9}'.format('width', 'A')
for branch in BRANCHES:
    header += '{:>11}{:>8}'.format(branch[:10], 'vs A')
print(header)

for width in widths:
    row = '{:>7.2f}{:>9.2f}'.format(width, A_KL.get(width, float('nan')))
    for branch in BRANCHES:
        entry = results.get(branch, {}).get(width)
        if entry is None:
            row += '{:>11}{:>8}'.format('-', '-')
            continue
        accuracy = 100.0 * (1.0 - entry[1])
        row += '{:>11.2f}{:>+8.2f}'.format(
            accuracy, accuracy - A_KL.get(width, accuracy))
    print(row)

print()
reference = sum(A_KL.values()) / len(A_KL)
print('{:22} mean {:.2f}   worst {:.2f}'.format(
    'A (reference)', reference, min(A_KL.values())))
for branch in BRANCHES:
    table = results.get(branch, {})
    if not table:
        continue
    accuracies = [100.0 * (1.0 - v[1]) for v in table.values()]
    mean = sum(accuracies) / len(accuracies)
    print('{:22} mean {:.2f}   worst {:.2f}   vs A {:+.2f}'.format(
        branch, mean, min(accuracies), mean - reference))

out = os.path.join(WORK, 'logs')
for branch in BRANCHES:
    log_dir = 'logs/cifar100_{}'.format(branch)
    if os.path.isdir(log_dir):
        shutil.copytree(log_dir, os.path.join(out, 'cifar100_' + branch),
                        dirs_exist_ok=True)
print('\ncheckpoints copied to', out)